# ⚡ Member 3 — Transform: PySpark Transformations
**Business Goal:** Apply meaningful transformations that generate actionable business insights.

**Pipeline Role:** RAW DATA → SPARK TRANSFORMS → `data/processed/*.parquet`

### Transformations Applied:
| # | Transformation | Business Impact |
|---|---------------|----------------|
| 1 | Population Density | Identify high-density markets for delivery |
| 2 | Currency Normalization | True USD revenue across all markets |
| 3 | Trip Profitability Score | Find unprofitable time slots |
| 4 | Peak Hour Classification | Optimize surge pricing windows |
| 5 | Market ROI Join | Rank global markets by logistics ROI |

In [10]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *
from pyspark.sql.window import Window
import os
from datetime import datetime

print('✅ PySpark imported')
print(f'📅 Run time: {datetime.now()}')

✅ PySpark imported
📅 Run time: 2026-05-12 12:17:31.673562


In [4]:
import os

os.environ["JAVA_HOME"] = r"C:\Java\jdk1.8.0_202"
os.environ["SPARK_HOME"] = r"C:\spark-3"
os.environ["HADOOP_HOME"] = r"C:\hadoop-3.2.4"
os.environ["PATH"] += r";C:\spark-3\bin;C:\hadoop-3.2.4\bin;C:\Java\jdk1.8.0_202\bin"

In [10]:
import os

os.environ["JAVA_HOME"] = r"C:\Java\jdk1.8.0_202"
os.environ["SPARK_HOME"] = r"C:\spark-3"
os.environ["HADOOP_HOME"] = r"C:\hadoop-3.2.4"

os.environ["PATH"] = (
    r"C:\spark-3\bin;"
    r"C:\hadoop-3.2.4\bin;"
    r"C:\Java\jdk1.8.0_202\bin;"
    + os.environ["PATH"]
)

In [6]:
import os
print(os.environ["SPARK_HOME"])
print(os.environ["JAVA_HOME"])

C:\spark-3
C:\Java\jdk1.8.0_202


In [2]:
pip install findspark

Note: you may need to restart the kernel to use updated packages.


In [4]:
import os
import findspark

os.environ["JAVA_HOME"] = r"C:\Java\jdk1.8.0_202"
os.environ["SPARK_HOME"] = r"C:\spark-3"
os.environ["HADOOP_HOME"] = r"C:\hadoop-3.2.4"

os.environ["PATH"] = (
    r"C:\spark-3\bin;"
    r"C:\hadoop-3.2.4\bin;"
    r"C:\Java\jdk1.8.0_202\bin;"
    + os.environ["PATH"]
)

findspark.init(r"C:\spark-3")

In [5]:
import os
print(os.environ["SPARK_HOME"])

C:\spark-3


In [11]:
# ── START SPARK SESSION ───────────────────────────────────────────
spark = SparkSession.builder \
    .appName('GlobalLogisticsETL') \
    .config('spark.sql.shuffle.partitions', '8') \
    .config('spark.driver.memory', '2g') \
    .getOrCreate()

spark.sparkContext.setLogLevel('ERROR')
print(f'✅ Spark started: version {spark.version}')

os.makedirs('data/processed', exist_ok=True)

✅ Spark started: version 3.5.1


## 📥 Load Raw Data

In [12]:
# ── LOAD ALL 3 RAW SOURCES ────────────────────────────────────────
print('📂 Loading raw data sources...')

df_countries = spark.read.parquet('data/raw/countries_raw.parquet')
df_taxi      = spark.read.parquet('data/raw/taxi_raw.parquet')
df_fx        = spark.read.parquet('data/raw/fx_rates_raw.parquet')

print(f'✅ Countries: {df_countries.count():,} rows')
print(f'✅ Taxi trips: {df_taxi.count():,} rows')
print(f'✅ FX rates:  {df_fx.count():,} rows')

📂 Loading raw data sources...
✅ Countries: 245 rows
✅ Taxi trips: 2,964,624 rows
✅ FX rates:  172 rows


## 🔧 Transform 1 — Countries: Population Density
**Why:** Population density tells us delivery cost per customer. Dense cities = efficient. Sparse = expensive.

In [13]:
# ── TRANSFORM 1: Population Density ──────────────────────────────
print('🔧 Transform 1: Population Density...')

df_countries_clean = df_countries \
    .filter(F.col('area_km2') > 0) \
    .filter(F.col('population') > 0) \
    .withColumn(
        'population_density',
        F.round(F.col('population') / F.col('area_km2'), 2)
    ) \
    .withColumn(
        'market_size_category',
        F.when(F.col('population') > 100_000_000, 'MEGA')
         .when(F.col('population') > 10_000_000,  'LARGE')
         .when(F.col('population') > 1_000_000,   'MEDIUM')
         .otherwise('SMALL')
    ) \
    .withColumn(
        'density_category',
        F.when(F.col('population_density') > 500, 'VERY_HIGH')
         .when(F.col('population_density') > 100, 'HIGH')
         .when(F.col('population_density') > 20,  'MEDIUM')
         .otherwise('LOW')
    )

print('\n📊 IMPACT: Market Size Distribution:')
df_countries_clean.groupBy('market_size_category').count().orderBy('count', ascending=False).show()

print('\n🌍 Top 10 Most Dense Markets (Best for delivery ROI):')
df_countries_clean.select('country_name','region','population_density','market_size_category') \
    .orderBy(F.col('population_density').desc()) \
    .show(10)

🔧 Transform 1: Population Density...

📊 IMPACT: Market Size Distribution:
+--------------------+-----+
|market_size_category|count|
+--------------------+-----+
|               SMALL|   84|
|               LARGE|   78|
|              MEDIUM|   67|
|                MEGA|   16|
+--------------------+-----+


🌍 Top 10 Most Dense Markets (Best for delivery ROI):
+------------+--------+------------------+--------------------+
|country_name|  region|population_density|market_size_category|
+------------+--------+------------------+--------------------+
|       Macau|    Asia|          22863.33|               SMALL|
|      Monaco|  Europe|          19021.29|               SMALL|
|   Singapore|    Asia|           8605.92|              MEDIUM|
|   Hong Kong|    Asia|           6818.39|              MEDIUM|
|   Gibraltar|  Europe|           6333.33|               SMALL|
|     Bahrain|    Asia|           2084.52|              MEDIUM|
|       Malta|  Europe|           1817.25|               SMALL|

## 🔧 Transform 2 — FX Rates: Currency Normalization
**Why:** Without normalization, 1000 ETB looks like 1000 USD. We need true USD values for real comparison.

In [14]:
# ── TRANSFORM 2: Currency Normalization ───────────────────────────
print('🔧 Transform 2: Currency Normalization...')

df_fx_clean = df_fx \
    .filter(F.col('rate_to_usd') > 0) \
    .withColumn(
        'local_to_usd_rate',
        F.round(F.lit(1.0) / F.col('rate_to_usd'), 6)
    ) \
    .withColumn(
        'currency_strength',
        F.when(F.col('local_to_usd_rate') >= 1.0,  'STRONG')
         .when(F.col('local_to_usd_rate') >= 0.1,  'MODERATE')
         .when(F.col('local_to_usd_rate') >= 0.01, 'WEAK')
         .otherwise('VERY_WEAK')
    ) \
    .withColumn(
        'example_100usd_in_local',
        F.round(F.lit(100.0) * F.col('rate_to_usd'), 2)
    )

# Join FX rates with Countries
df_countries_fx = df_countries_clean.join(
    df_fx_clean.select('currency_code', 'local_to_usd_rate', 'currency_strength', 'rate_to_usd'),
    on='currency_code',
    how='left'
).fillna({'local_to_usd_rate': 1.0, 'currency_strength': 'UNKNOWN'})

print('\n📊 IMPACT: Currency Strength by Region:')
df_countries_fx.groupBy('region', 'currency_strength').count().orderBy('region', 'count').show(20)

print('\n💡 Business Insight: Effective purchasing power per region')
df_countries_fx.groupBy('region') \
    .agg(
        F.round(F.avg('local_to_usd_rate'), 4).alias('avg_local_to_usd'),
        F.count('*').alias('num_countries')
    ).orderBy('avg_local_to_usd', ascending=False).show()

🔧 Transform 2: Currency Normalization...

📊 IMPACT: Currency Strength by Region:
+---------+-----------------+-----+
|   region|currency_strength|count|
+---------+-----------------+-----+
|   Africa|         MODERATE|    3|
|   Africa|           STRONG|    3|
|   Africa|             WEAK|   17|
|   Africa|        VERY_WEAK|   35|
| Americas|             WEAK|    6|
| Americas|        VERY_WEAK|    9|
| Americas|         MODERATE|   20|
| Americas|           STRONG|   20|
|Antarctic|          UNKNOWN|    1|
|Antarctic|           STRONG|    1|
|     Asia|           STRONG|    5|
|     Asia|             WEAK|   10|
|     Asia|         MODERATE|   14|
|     Asia|        VERY_WEAK|   21|
|   Europe|        VERY_WEAK|    2|
|   Europe|             WEAK|    7|
|   Europe|         MODERATE|   10|
|   Europe|           STRONG|   34|
|  Oceania|          UNKNOWN|    1|
|  Oceania|        VERY_WEAK|    4|
+---------+-----------------+-----+
only showing top 20 rows


💡 Business Insight: Effectiv

## 🔧 Transform 3 — Taxi: Trip Profitability
**Why:** Not all trips make money. We identify unprofitable trips and time slots.

In [15]:
# ── TRANSFORM 3: Trip Profitability ───────────────────────────────
print('🔧 Transform 3: Trip Profitability Analysis...')

df_taxi_clean = df_taxi \
    .filter(F.col('fare_amount') > 0) \
    .filter(F.col('trip_distance') > 0) \
    .filter(F.col('fare_amount') < 500) \
    .filter(F.col('trip_distance') < 100) \
    .withColumn(
        'revenue_per_mile',
        F.round(F.col('fare_amount') / F.col('trip_distance'), 2)
    ) \
    .withColumn(
        'is_profitable',
        F.when(F.col('fare_amount') >= 5.0, True).otherwise(False)
    ) \
    .withColumn(
        'profitability_tier',
        F.when(F.col('fare_amount') >= 30, 'HIGH_VALUE')
         .when(F.col('fare_amount') >= 10, 'STANDARD')
         .when(F.col('fare_amount') >= 5,  'LOW_MARGIN')
         .otherwise('UNPROFITABLE')
    ) \
    .withColumn('pickup_hour', F.hour('tpep_pickup_datetime')) \
    .withColumn('pickup_day',  F.dayofweek('tpep_pickup_datetime')) \
    .withColumn('pickup_date', F.to_date('tpep_pickup_datetime'))

total = df_taxi_clean.count()
unprofitable = df_taxi_clean.filter(~F.col('is_profitable')).count()
pct = (unprofitable / total) * 100

print(f'\n📊 IMPACT — Profitability Discovery:')
print(f'   Total valid trips:    {total:,}')
print(f'   Unprofitable trips:   {unprofitable:,} ({pct:.1f}%)')
print(f'   → ACTION: These {pct:.0f}% of trips are costing the business money!')

print('\n📊 Revenue Tier Breakdown:')
df_taxi_clean.groupBy('profitability_tier').count() \
    .withColumn('percentage', F.round(F.col('count') / total * 100, 1)) \
    .orderBy('count', ascending=False).show()

🔧 Transform 3: Trip Profitability Analysis...

📊 IMPACT — Profitability Discovery:
   Total valid trips:    2,869,637
   Unprofitable trips:   44,122 (1.5%)
   → ACTION: These 2% of trips are costing the business money!

📊 Revenue Tier Breakdown:
+------------------+-------+----------+
|profitability_tier|  count|percentage|
+------------------+-------+----------+
|          STANDARD|1587858|      55.3|
|        LOW_MARGIN| 831062|      29.0|
|        HIGH_VALUE| 406595|      14.2|
|      UNPROFITABLE|  44122|       1.5|
+------------------+-------+----------+



## 🔧 Transform 4 — Taxi: Peak Hour Analysis
**Why:** Surge pricing should be based on real demand data, not guesswork.

In [16]:
# ── TRANSFORM 4: Peak Hour Classification ─────────────────────────
print('🔧 Transform 4: Peak Hour Classification...')

df_taxi_hours = df_taxi_clean \
    .withColumn(
        'time_period',
        F.when((F.col('pickup_hour') >= 7)  & (F.col('pickup_hour') <= 9),  'MORNING_RUSH')
         .when((F.col('pickup_hour') >= 17) & (F.col('pickup_hour') <= 19), 'EVENING_RUSH')
         .when((F.col('pickup_hour') >= 22) | (F.col('pickup_hour') <= 4),  'LATE_NIGHT')
         .when((F.col('pickup_hour') >= 11) & (F.col('pickup_hour') <= 14), 'LUNCH_HOUR')
         .otherwise('OFF_PEAK')
    )

# Aggregate by hour
df_hourly = df_taxi_hours.groupBy('pickup_hour', 'time_period') \
    .agg(
        F.count('*').alias('trip_count'),
        F.round(F.avg('fare_amount'), 2).alias('avg_fare_usd'),
        F.round(F.avg('revenue_per_mile'), 2).alias('avg_rev_per_mile'),
        F.round(F.sum('fare_amount'), 2).alias('total_revenue_usd'),
        F.round(F.avg(F.col('is_profitable').cast('int')) * 100, 1).alias('profitable_pct')
    ).orderBy('pickup_hour')

print('\n📊 IMPACT — Hourly Revenue Analysis (KEY FINDING):')
df_hourly.show(24)

print('\n💡 Business Insight: Time Period Performance:')
df_taxi_hours.groupBy('time_period') \
    .agg(
        F.count('*').alias('trips'),
        F.round(F.avg('fare_amount'), 2).alias('avg_fare'),
        F.round(F.avg(F.col('is_profitable').cast('int')) * 100, 1).alias('profitable_pct')
    ).orderBy('avg_fare', ascending=False).show()

🔧 Transform 4: Peak Hour Classification...

📊 IMPACT — Hourly Revenue Analysis (KEY FINDING):
+-----------+------------+----------+------------+----------------+-----------------+--------------+
|pickup_hour| time_period|trip_count|avg_fare_usd|avg_rev_per_mile|total_revenue_usd|profitable_pct|
+-----------+------------+----------+------------+----------------+-----------------+--------------+
|          0|  LATE_NIGHT|     75239|       19.68|           11.28|       1480986.39|          98.2|
|          1|  LATE_NIGHT|     50480|       17.73|           11.88|        895048.01|          98.0|
|          2|  LATE_NIGHT|     34960|       16.62|           11.28|         581041.3|          97.7|
|          3|  LATE_NIGHT|     22942|       18.54|           12.83|        425267.89|          97.6|
|          4|  LATE_NIGHT|     15275|       23.42|           13.23|        357790.82|          97.6|
|          5|    OFF_PEAK|     17493|        27.5|           15.65|        481029.96|          98.

## 🔧 Transform 5 — Final Join: Market ROI Score
**Why:** Combines all 3 sources into one unified insight — which global markets are best for expansion?

In [17]:
# ── TRANSFORM 5: Market ROI Score (Big Join) ──────────────────────
print('🔧 Transform 5: Global Market ROI Score...')

# Regional taxi performance summary
df_taxi_summary = df_taxi_hours.agg(
    F.round(F.avg('fare_amount'), 2).alias('global_avg_fare'),
    F.round(F.avg('revenue_per_mile'), 2).alias('global_avg_rev_mile'),
    F.round(F.avg(F.col('is_profitable').cast('int')) * 100, 1).alias('global_profitable_pct')
).first()

global_avg_fare    = df_taxi_summary['global_avg_fare']
global_profitable  = df_taxi_summary['global_profitable_pct']

print(f'   Global avg fare:       ${global_avg_fare}')
print(f'   Global profitable pct: {global_profitable}%')

# Build regional market score
df_market_roi = df_countries_fx \
    .withColumn(
        'logistics_cost_index',
        F.round(
            F.when(F.col('population_density') > 0,
                   F.lit(1000.0) / F.col('population_density'))
             .otherwise(F.lit(999.0)),
            2
        )
    ) \
    .withColumn(
        'currency_stability_score',
        F.when(F.col('currency_strength') == 'STRONG',    F.lit(10.0))
         .when(F.col('currency_strength') == 'MODERATE',  F.lit(7.0))
         .when(F.col('currency_strength') == 'WEAK',      F.lit(4.0))
         .otherwise(F.lit(2.0))
    ) \
    .withColumn(
        'market_score',
        F.round(
            (F.log1p(F.col('population')) * 0.3) +
            (F.col('currency_stability_score') * 0.4) +
            (F.lit(10.0) / (F.col('logistics_cost_index') + 1) * 0.3),
            2
        )
    ) \
    .withColumn(
        'expansion_recommendation',
        F.when(F.col('market_score') >= 8, 'EXPAND_NOW')
         .when(F.col('market_score') >= 5, 'MONITOR')
         .otherwise('HOLD')
    )

print('\n📊 IMPACT — Top Markets for Expansion:')
df_market_roi.select(
    'country_name', 'region', 'market_size_category',
    'currency_strength', 'market_score', 'expansion_recommendation'
).orderBy(F.col('market_score').desc()).show(15)

print('\n🌍 Expansion Recommendations by Region:')
df_market_roi.groupBy('region', 'expansion_recommendation').count() \
    .orderBy('region').show(30)

🔧 Transform 5: Global Market ROI Score...
   Global avg fare:       $18.49
   Global profitable pct: 98.5%

📊 IMPACT — Top Markets for Expansion:
+--------------+--------+--------------------+-----------------+------------+------------------------+
|  country_name|  region|market_size_category|currency_strength|market_score|expansion_recommendation|
+--------------+--------+--------------------+-----------------+------------+------------------------+
|       Bahrain|    Asia|              MEDIUM|           STRONG|       10.31|              EXPAND_NOW|
|     Singapore|    Asia|              MEDIUM|         MODERATE|       10.17|              EXPAND_NOW|
|     Hong Kong|    Asia|              MEDIUM|         MODERATE|       10.16|              EXPAND_NOW|
|United Kingdom|  Europe|               LARGE|           STRONG|       10.08|              EXPAND_NOW|
|       Germany|  Europe|               LARGE|           STRONG|       10.04|              EXPAND_NOW|
|        Monaco|  Europe|     

## 💾 Save All Transformed Data

In [18]:
# ── SAVE ALL TRANSFORMED DATASETS ────────────────────────────────
print('💾 Saving transformed datasets...')

# 1. Countries with density + FX
df_market_roi.toPandas().to_parquet('data/processed/countries_transformed.parquet', index=False)
print('✅ Saved: countries_transformed.parquet')

# 2. FX rates cleaned
df_fx_clean.toPandas().to_parquet('data/processed/fx_rates_transformed.parquet', index=False)
print('✅ Saved: fx_rates_transformed.parquet')

# 3. Taxi trips with profitability
df_taxi_hours.toPandas().to_parquet('data/processed/taxi_transformed.parquet', index=False)
print('✅ Saved: taxi_transformed.parquet')

# 4. Hourly summary (small, for dashboard)
df_hourly.toPandas().to_parquet('data/processed/taxi_hourly_summary.parquet', index=False)
print('✅ Saved: taxi_hourly_summary.parquet')

print('\n🏁 Member 3 COMPLETE — All transformed data ready for Member 4 (DuckDB Load)')
spark.stop()

💾 Saving transformed datasets...
✅ Saved: countries_transformed.parquet
✅ Saved: fx_rates_transformed.parquet
✅ Saved: taxi_transformed.parquet
✅ Saved: taxi_hourly_summary.parquet

🏁 Member 3 COMPLETE — All transformed data ready for Member 4 (DuckDB Load)
